[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cafzal/frontier/blob/main/examples/cuopt_bidirectional/cuopt_bidirectional.ipynb)

# Frontier × cuOpt — bidirectional benefits, across two problems

**The question.** Does wrapping NVIDIA **cuOpt** inside Frontier's multi-objective engine help
*both* tools? We test it honestly on two problems — a **portfolio** (allocate, QP) and a
**capital-project portfolio** (select, MILP) — by running each **three ways**:

- **cuOpt alone** — cuOpt is single-objective, so you commit to one weighting and get **one
  plan**, blind to what you traded away.
- **Frontier alone (NSGA-II)** — an *approximate* multi-objective frontier, no optimality guarantees.
- **Frontier + cuOpt** — the frontier, **optimal-to-tolerance at each point**, with cuOpt's gap-bounded solves + QP duals.

**What to look for (the honest bidirectional read):**
- **Frontier → cuOpt (large, structural):** the lone plan vs the frontier. cuOpt alone gives one
  point; Frontier turns it into an explorable tradeoff surface — a capability cuOpt lacks.
- **cuOpt → Frontier (scale-dependent):** on the convex portfolio NSGA matches the coverage, so cuOpt's
  edge is optimal-to-tolerance corners + QP shadow prices (see the dual at the end). On the 120-project
  MILP the exact frontier covers materially more — the pairing wins on a large, irregular combinatorial problem.

**Caveat, stated up front:** these datasets are **synthetic**. The Frontier→cuOpt benefit is
structural (realism-independent); the cuOpt→cuOpt coverage parity may partly be a "well-conditioned
random data" artifact — we don't claim cuOpt beats NSGA on coverage.

**Requirements.** cuOpt needs **Linux + an NVIDIA GPU** — pick a Colab GPU runtime
(*Runtime → Change runtime type → T4 GPU*) and *Run all*. Engine + data clone from the public repo.

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print(out.stdout.strip() or "no nvidia-smi output")
except FileNotFoundError:
    print("No NVIDIA GPU detected - cuOpt cannot run. In Colab: Runtime -> Change runtime type -> GPU (T4).")

In [ ]:
# Bootstrap: clone Frontier (public) + install cuOpt + engine deps. First run ~2-3 min.
import os, subprocess, sys
REPO = "/content/frontier"
if not os.path.isdir(REPO):
    rc = subprocess.run(["git", "clone", "-q", "https://github.com/cafzal/frontier.git", REPO],
                        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}).returncode
    if rc != 0 or not os.path.isdir(REPO):
        raise RuntimeError("Could not clone github.com/cafzal/frontier - check the connection, then re-run.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--extra-index-url", "https://pypi.nvidia.com", "cuopt-cu12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pymoo>=0.6", "pydantic>=2.0", "scipy>=1.11", "pandas>=2.0", "matplotlib>=3.7"], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("Frontier on path:", REPO in sys.path)

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
from engine.models import Problem
from engine.optimizer import optimize
from solvers import cuopt_backend as cb
# Confirms the GPU solver is live (fails if not on a cuOpt runtime):
from cuopt.linear_programming.problem import Problem as CuProblem
print("cuOpt import OK - GPU solver live.")

In [ ]:
REPO = "/content/frontier"

def load_example(name):
    p = json.load(open(f"{REPO}/examples/{name}/problem.json"))
    s = json.load(open(f"{REPO}/examples/{name}/scores.json"))
    return Problem(**{**p, **s})

def _points(run, objs):
    return np.array([[s.objective_values[o.name] for o in objs] for s in run.solutions])

def cuopt_alone(problem):
    """What cuOpt gives WITHOUT Frontier: ONE optimal point for one scalarization
    (optimize the first objective), ignoring the multi-objective tradeoff."""
    objs = problem.objectives
    if problem.approach.value == "binary":
        n, names, S, dirs, mc = cb._build_milp_data(problem)
        sel, ok = cb._solve_milp_cuopt(dirs[0] * S[:, 0], [], mc, n)
        return {o.name: float(S[:, j] @ sel) for j, o in enumerate(objs)}
    score = cb._opt._build_score_matrix(problem)
    im = cb._opt._build_interaction_matrices(problem)
    ri, _ = cb._resolve_objective_roles(problem)
    cov = cb._nearest_psd(im[ri])
    cp = cb._opt._parse_constraints(problem)
    mw = (cp["max_allocation"] / 100.0) if cp.get("max_allocation") else None
    lin = cb._resolve_linear_objectives(problem)
    mu = score[:, lin[0]]
    groups = cb._group_limits(problem)
    support = None
    if groups:                      # group-feasible support so the single solve is valid
        maximize = objs[lin[0]].direction.value == "maximize"
        support = np.array([i for grp, gmax in groups
                            for i in sorted(grp, key=lambda i: mu[i], reverse=maximize)[:gmax]])
    w, ok = cb._solve_qp_cuopt(cov, mu, None, True, mw, support=support)
    prop = cb._opt._ProportionalProblem(n_options=len(problem.options), score_matrix=score,
                                        objectives=objs, interaction_matrices=im, **cp)
    W = (w * 100).reshape(1, -1)
    return {o.name: float(prop._aggregate_objective(W, j)[0]) for j, o in enumerate(objs)}

def three_way(name, ex, xi, yi):
    problem = load_example(ex)
    objs = problem.objectives
    os.environ.pop("FRONTIER_SOLVER", None)
    nsga = _points(optimize(problem, seed=42), objs)
    os.environ["FRONTIER_SOLVER"] = "cuopt"
    integ = _points(optimize(problem, seed=42), objs)
    a = cuopt_alone(problem)
    ax_, ay_ = objs[xi].name, objs[yi].name
    print(f"{name}: cuOpt-alone = 1 point | Frontier alone (NSGA) = {len(nsga)} pts | "
          f"Frontier + cuOpt = {len(integ)} pts")
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.scatter(nsga[:, xi], nsga[:, yi], s=42, color="darkorange", alpha=0.6,
               label=f"Frontier alone / NSGA ({len(nsga)})")
    ax.scatter(integ[:, xi], integ[:, yi], s=46, facecolor="none", edgecolor="navy",
               linewidth=1.4, zorder=3, label=f"Frontier + cuOpt ({len(integ)})")
    ax.scatter([a[ax_]], [a[ay_]], s=260, marker="*", color="crimson", edgecolor="k",
               zorder=5, label="cuOpt alone (1 point)")
    ax.set_xlabel(ax_); ax.set_ylabel(ay_)
    ax.set_title(f"{name}: cuOpt alone is ONE point - Frontier makes it a frontier")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

def scatter3d(name, ex, xi=0, yi=1, zi=2):
    """Three-objective view: show all three objectives as a 3-D scatter. A 2-D slice hides
    the third axis and makes trades into it look 'interior'; the 3-D cloud is honest. cuOpt
    alone is ONE point in the cloud; Frontier + cuOpt is the efficient surface."""
    problem = load_example(ex); objs = problem.objectives
    os.environ.pop("FRONTIER_SOLVER", None)
    nsga = _points(optimize(problem, seed=42), objs)
    os.environ["FRONTIER_SOLVER"] = "cuopt"
    integ = _points(optimize(problem, seed=42), objs)
    a = cuopt_alone(problem)
    ax_, ay_, az_ = objs[xi].name, objs[yi].name, objs[zi].name
    print(f"{name}: cuOpt-alone = 1 point | Frontier alone (NSGA) = {len(nsga)} pts | "
          f"Frontier + cuOpt = {len(integ)} pts  ({len(objs)} objectives)")
    fig = plt.figure(figsize=(8, 6.5))
    ax = fig.add_subplot(projection="3d")
    ax.scatter(nsga[:, xi], nsga[:, yi], nsga[:, zi], s=26, color="darkorange", alpha=0.45,
               label=f"Frontier alone / NSGA ({len(nsga)})")
    ax.scatter(integ[:, xi], integ[:, yi], integ[:, zi], s=44, color="navy", alpha=0.9,
               depthshade=False, label=f"Frontier + cuOpt ({len(integ)})")
    ax.scatter([a[ax_]], [a[ay_]], [a[az_]], s=340, marker="*", color="crimson", edgecolor="k",
               depthshade=False, label="cuOpt alone (1 point)")
    ax.set_xlabel(f"{ax_} ({'min' if objs[xi].direction.value == 'minimize' else 'max'})")
    ax.set_ylabel(f"{ay_} ({'min' if objs[yi].direction.value == 'minimize' else 'max'})")
    ax.set_zlabel(f"{az_} ({'min' if objs[zi].direction.value == 'minimize' else 'max'})")
    ax.view_init(elev=20, azim=-58)
    ax.set_title(f"{name}: all 3 objectives - cuOpt alone is ONE point in the cloud")
    ax.legend(loc="upper left", fontsize=8); plt.tight_layout(); plt.show()

def parallel_coords(name, ex):
    """Many-objective view: a 2-D scatter hides axes, so plot ALL objectives as parallel
    coordinates (each normalized so higher = better). cuOpt alone is ONE bold polyline;
    Frontier + cuOpt is the whole sheaf of efficient plans - the same thesis as the scatters,
    for a problem with too many objectives to project to a plane."""
    problem = load_example(ex); objs = problem.objectives
    os.environ.pop("FRONTIER_SOLVER", None)
    nsga = _points(optimize(problem, seed=42), objs)
    os.environ["FRONTIER_SOLVER"] = "cuopt"
    integ = _points(optimize(problem, seed=42), objs)
    a = np.array([cuopt_alone(problem)[o.name] for o in objs])
    allpts = np.vstack([nsga, integ, a.reshape(1, -1)])
    lo = allpts.min(0); hi = allpts.max(0); span = np.where(hi > lo, hi - lo, 1.0)
    def norm(P):                      # to [0,1], flipped for minimize so up = better everywhere
        Z = (np.atleast_2d(P) - lo) / span
        for j, o in enumerate(objs):
            if o.direction.value == "minimize":
                Z[:, j] = 1.0 - Z[:, j]
        return Z
    xs = np.arange(len(objs))
    print(f"{name}: cuOpt-alone = 1 plan | Frontier alone (NSGA) = {len(nsga)} pts | "
          f"Frontier + cuOpt = {len(integ)} pts  ({len(objs)} objectives)")
    fig, ax = plt.subplots(figsize=(1.7 * len(objs) + 2, 5.6))
    for row in norm(nsga):
        ax.plot(xs, row, color="darkorange", alpha=0.22, lw=0.8, zorder=1)
    for row in norm(integ):
        ax.plot(xs, row, color="navy", alpha=0.45, lw=1.0, zorder=2)
    ax.plot(xs, norm(a)[0], color="crimson", lw=3.2, marker="o", ms=9, zorder=5)
    ax.plot([], [], color="darkorange", lw=2, label=f"Frontier alone / NSGA ({len(nsga)})")
    ax.plot([], [], color="navy", lw=2, label=f"Frontier + cuOpt ({len(integ)})")
    ax.plot([], [], color="crimson", lw=3.2, marker="o", ms=9, label="cuOpt alone (1 plan)")
    for x in xs:
        ax.axvline(x, color="gray", alpha=0.3, lw=0.8)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{o.name}\n({'min' if o.direction.value == 'minimize' else 'max'})" for o in objs])
    ax.set_ylim(-0.05, 1.08); ax.set_ylabel("normalized score (higher = better on every axis)")
    ax.set_title(f"{name}: cuOpt alone is ONE plan - Frontier maps the whole tradeoff")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False)
    ax.grid(alpha=0.25, axis="y"); plt.tight_layout(); plt.show()

def coverage_showcase(name, ex, xi, yi):
    """At scale the efficient frontier is large + irregular, and three rungs separate by hypervolume -
    all feasible: NSGA-alone (pure heuristic) < Frontier + cuOpt (the integration - an EA evolving
    scalarizations, each solved to a 0.1% gap by cuOpt's exact MILP) < cuOpt's exact frontier (the
    ceiling). We report each rung's share of that exact hypervolume - the pairing reaches frontier the
    heuristic leaves uncovered, while the direct sweep is the most solve-efficient ceiling."""
    from pymoo.indicators.hv import HV
    problem = load_example(ex); objs = problem.objectives
    n, names, S, dirs, mc = cb._build_milp_data(problem)
    cmin, cmax = S.min(0), S.max(0); Snorm = (S - cmin) / np.where(cmax > cmin, cmax - cmin, 1.0)
    os.environ.pop("FRONTIER_SOLVER", None)
    nsga = _points(optimize(problem, seed=42), objs)             # pure heuristic, no cuOpt
    os.environ["FRONTIER_SOLVER"] = "cuopt"
    integ = _points(optimize(problem, seed=42), objs)            # the integration: EA over exact cuOpt MILP
    rng = np.random.default_rng(0)
    W = [w for w in np.eye(len(objs))] + [w / w.sum() for w in rng.random((300, len(objs)))]
    seen, exact = set(), []
    for w in W:
        sel, ok = cb._solve_milp_cuopt(Snorm @ (np.asarray(w) * dirs), [], mc, n)   # weighted-sum scalarization
        if not ok:
            continue
        key = tuple(round(float(S[:, j] @ sel), 3) for j in range(len(objs)))
        if key not in seen:
            seen.add(key); exact.append(list(key))
    exact = np.array(exact)
    em, bm, gm = exact * dirs, nsga * dirs, integ * dirs         # minimize space for hypervolume
    ceil_pts = np.vstack([em, gm])   # exact-frontier ceiling = ALL exact MILP optima (sweep + EA); HV is
    U = np.vstack([ceil_pts, bm])    # monotone, so integ is guaranteed <= 100% without nondom filtering
    lo, hi = U.min(0), U.max(0); span = np.where(hi > lo, hi - lo, 1.0)
    ref = np.full(len(objs), 1.1)
    hv = lambda P: HV(ref_point=ref)((P - lo) / span)
    hv_exact = hv(ceil_pts); hv_nsga = hv(bm) / hv_exact; hv_integ = hv(gm) / hv_exact
    print(f"{name} HV ladder: NSGA-alone {len(nsga)} pts = {hv_nsga:.0%}  <  "
          f"Frontier+cuOpt EA {len(integ)} pts = {hv_integ:.0%}  <  cuOpt exact frontier {len(exact)} pts = 100%")
    fig, ax = plt.subplots(figsize=(7.8, 5.6))
    ax.scatter(nsga[:, xi], nsga[:, yi], s=46, color="darkorange", alpha=0.5, label=f"NSGA-alone ({len(nsga)}) = {hv_nsga:.0%}")
    ax.scatter(integ[:, xi], integ[:, yi], s=34, color="seagreen", alpha=0.8, label=f"Frontier + cuOpt EA ({len(integ)}) = {hv_integ:.0%}")
    ax.scatter(exact[:, xi], exact[:, yi], s=16, color="navy", alpha=0.85, label=f"cuOpt exact frontier ({len(exact)}) = 100%")
    ax.set_xlabel(objs[xi].name); ax.set_ylabel(objs[yi].name)
    ax.set_title(f"{name}: the pairing climbs the ladder - NSGA {hv_nsga:.0%} < EA+cuOpt {hv_integ:.0%} < exact 100%")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 1 - Portfolio (allocate across 30 ETFs - QP)

Return / volatility (quadratic) / yield - **three** objectives, so we show all three as a 3-D scatter.
cuOpt alone returns the single minimum-risk portfolio (the star); Frontier turns its QP solve into the
whole risk/return/yield surface. (In a 2-D slice the Frontier+cuOpt points can look 'interior' only
because they trade into the hidden yield axis - the 3-D cloud shows they don't.)

In [ ]:
scatter3d("Portfolio", "investment_portfolio", 0, 1, 2)

## 2 - Capital projects at scale (select from 120 - MILP)

NPV / cost / risk / strategic-fit over **120 projects** under a hard budget + per-category caps +
dependencies + exclusions. At this scale the frontier is large and irregular, and three rungs separate
cleanly by hypervolume: **NSGA-alone** (the pure heuristic) covers part of it; **Frontier + cuOpt** (the
integration - an EA evolving scalarizations, each solved exactly by cuOpt's MILP) climbs higher;
**cuOpt's exact frontier** is the ceiling. **That ladder is where the pairing pays off — a hard
combinatorial problem, not a convex one** (contrast the portfolio above, where the heuristic already
matches the integration). The EA isn't the most solve-efficient explorer here — the direct sweep is —
but it reaches frontier the heuristic can't.

In [ ]:
coverage_showcase("Capital projects", "capital_project_selection_120", 0, 1)

## 3 - cuOpt -> Frontier: the narrow benefit (QP shadow prices)

Coverage washes (NSGA matches the integration above), but cuOpt brings something NSGA cannot:
**shadow prices** - the convex-QP dual, to solver tolerance. On the portfolio, the dual of the return
constraint is the *shadow price of return* - the marginal risk you accept for one more unit of return -
rising monotonically along the efficient edge. (Duals are a continuous-QP object; cuOpt returns none
for a MILP.)

In [ ]:
problem = load_example("investment_portfolio")
score = cb._opt._build_score_matrix(problem); im = cb._opt._build_interaction_matrices(problem)
ri, _ = cb._resolve_objective_roles(problem); lin = cb._resolve_linear_objectives(problem)
cov = cb._nearest_psd(im[ri]); mu = score[:, lin[0]]
cp = cb._opt._parse_constraints(problem); mw = (cp["max_allocation"]/100.0) if cp.get("max_allocation") else None
w_mv, _ = cb._solve_qp_cuopt(cov, mu, None, True, mw)
r_lo, r_hi = float(mu @ w_mv), float(mu.max())
rets, duals = [], []
for tgt in np.linspace(r_lo, r_hi * 0.999, 25):
    n = len(mu); p = CuProblem("dual_demo")
    wv = [p.addVariable(lb=0.0, ub=mw if mw else 1.0, name=f"w{i}") for i in range(n)]
    quad = None
    for i in range(n):
        for j in range(n):
            c = float(cov[i, j])
            if abs(c) > 1e-12:
                t = c * wv[i] * wv[j]; quad = t if quad is None else quad + t
    from cuopt.linear_programming.problem import MINIMIZE
    p.setObjective(quad, sense=MINIMIZE)
    p.addConstraint(sum(wv) == 1, name="budget")
    rc = p.addConstraint(sum(float(mu[i]) * wv[i] for i in range(n)) >= float(tgt), name="return_target")
    p.solve()
    if getattr(p.Status, "name", "") not in ("Optimal", "PrimalFeasible"):
        continue
    rets.append(float(mu @ np.array([wv[i].Value for i in range(n)]))); duals.append(abs(float(rc.DualValue)))
fig, ax = plt.subplots(); ax.plot(rets, duals, "o-", color="purple", lw=1.5)
ax.set_xlabel("Expected Return achieved (%)"); ax.set_ylabel("Shadow price d(variance)/d(return)")
ax.set_title("cuOpt dual - the marginal risk cost of return (NSGA cannot produce this)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"shadow price {min(duals):.3f} -> {max(duals):.3f} as return rises - the QP return-constraint dual (to solver tolerance).")

## 4 - Conclusions: the integration is bidirectional (and honest)

Across both problems, one picture repeats:

- **Frontier -> cuOpt (the large, structural benefit).** The single cuOpt-alone plan (the red star) is
  everything cuOpt gives you *alone*: **one plan**, for one weighting you had to guess, with no view of
  the tradeoff. Frontier turns cuOpt's single optimal solve into an **explorable frontier** - the
  multi-objective decision layer a single-objective solver structurally lacks. This holds on every
  problem and is realism-independent.
- **cuOpt -> Frontier (two regimes).** On the **convex portfolio**, NSGA already matches the integration
  on *coverage* — cuOpt's edge there is **optimality to tolerance** (each point solved to a 0.1% MILP
  gap / PDLP tolerance — a feasible near-optimum, not a certified one) plus **QP shadow prices** (3),
  the convex-QP dual NSGA cannot produce. On the **120-project MILP** it changes: the exact frontier
  **covers materially more** than NSGA's fixed-resolution population (the hypervolume gap above). So the
  honest rule is scale- and structure-dependent — the heuristic ties on smooth/convex problems; the
  exact solver pulls ahead on large, irregular combinatorial ones.

**Most compelling for the cuOpt team:** the **120-project MILP** — at scale the exact frontier covers
materially more of the tradeoff surface than the heuristic, the one place the pairing demonstrably wins
— together with the **QP shadow prices** (3) on the portfolio. The portfolio is the convex case where
NSGA already matches the coverage, so its job is the familiar on-ramp + the duals, not a superiority claim.

**One-line thesis:** *cuOpt makes each point optimal (to tolerance); Frontier makes the set a decision.*
The frontier itself - not just one optimal point - is what the integration adds, and per-point
optimality + shadow prices are what cuOpt adds back.